# ReceiptGuard-ML Kaggle Training Notebook

This notebook runs the complete ReceiptGuard-ML pipeline on Kaggle.
Each cell imports from existing pipeline modules - no logic duplication.

## Cell 1: Setup & Install Dependencies

In [ ]:
# Install required packages for Kaggle environment
!pip install -q torch torchvision transformers accelerate
!pip install -q Pillow numpy scikit-learn pandas tqdm matplotlib seaborn
!pip install -q tensorboard python-dotenv pytesseract

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

## Cell 2: Mount SROIE2019 Dataset

In [ ]:
import os
import sys
from pathlib import Path

# Add project source to path
sys.path.insert(0, '/kaggle/input/receiptguard-ml/src')
sys.path.insert(0, '/kaggle/input/receiptguard-ml')

# Kaggle dataset paths
KAGGLE_INPUT_PATH = Path('/kaggle/input')
RECEIPTGUARD_PATH = KAGGLE_INPUT_PATH / 'receiptguard-ml'
SROIE2019_PATH = KAGGLE_INPUT_PATH / 'sroie2019'  # Adjust based on your dataset name

# Working directory for outputs
WORKING_PATH = Path('/kaggle/working')
WORKING_PATH.mkdir(parents=True, exist_ok=True)

# Determine dataset path
if SROIE2019_PATH.exists():
    RAW_DATA_PATH = SROIE2019_PATH / 'SROIE2019'
    print(f"SROIE2019 dataset found at: {RAW_DATA_PATH}")
else:
    # Fallback: look for dataset in receiptguard-ml input
    RAW_DATA_PATH = RECEIPTGUARD_PATH / 'dataset' / 'raw' / 'SROIE2019'
    print(f"Using dataset at: {RAW_DATA_PATH}")

# Verify dataset structure
if RAW_DATA_PATH.exists():
    splits = ['train', 'test']
    for split in splits:
        split_path = RAW_DATA_PATH / split
        if split_path.exists():
            print(f"  ✓ {split}/ directory exists")
        else:
            print(f"  ✗ {split}/ directory missing")
else:
    print(f"WARNING: Dataset not found at {RAW_DATA_PATH}")
    print("Available paths:")
    for p in KAGGLE_INPUT_PATH.iterdir():
        print(f"  - {p.name}")

## Cell 3: Run Preprocessing Pipeline

In [ ]:
from pipelines.preprocessing_pipeline import (
    run_preprocessing_pipeline,
    PreprocessingConfig
)

# Define preprocessing configuration
preprocess_config = PreprocessingConfig(
    raw_data_path=str(RAW_DATA_PATH),
    processed_data_path=str(WORKING_PATH / 'processed'),
    splits=['train', 'test'],
    verify_images=True
)

print("Starting preprocessing...")
preprocess_summary = run_preprocessing_pipeline(preprocess_config)

print(f"\nPreprocessing complete!")
print(f"Total samples: {preprocess_summary.get('total_samples', 0)}")
print(f"Failed samples: {preprocess_summary.get('failed_samples_count', 0)}")
print(f"Output directory: {preprocess_config.processed_data_path}")

## Cell 4: Training with Kaggle-Optimized Config (batch_size=16, GPU)

In [ ]:
from pipelines.model_training_pipeline import (
    run_training_pipeline,
    TrainingConfig
)

# Kaggle-optimized training configuration
# GPU allows larger batch size for faster training
training_config = TrainingConfig(
    model_path=str(RAW_DATA_PATH / 'layoutlm-base-uncased'),
    num_labels=9,  # O, B-COMPANY, I-COMPANY, B-DATE, I-DATE, B-ADDRESS, I-ADDRESS, B-TOTAL, I-TOTAL
    dropout=0.1,
    output_dir=str(WORKING_PATH / 'checkpoints'),
    num_epochs=15,
    batch_size=16,  # Optimized for Kaggle GPU (T4 or P100)
    max_length=512,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    seed=42,
    data_path=str(RAW_DATA_PATH)
)

print("Starting training with Kaggle-optimized config...")
print(f"Batch size: {training_config.batch_size}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Run training
training_summary = run_training_pipeline(training_config)

# Print results
if training_summary.get('status') == 'completed':
    print(f"\nTraining completed successfully!")
    print(f"Best eval loss: {training_summary.get('best_eval_loss', 'N/A'):.4f}")
    print(f"Best checkpoint: {training_summary.get('best_checkpoint', 'N/A')}")
    print(f"Total epochs: {training_summary.get('epochs_trained', 'N/A')}")
else:
    print(f"Training failed: {training_summary.get('error', 'Unknown error')}")

## Cell 5: Evaluation Pipeline

In [ ]:
from pipelines.evaluation_pipeline import (
    run_evaluation_pipeline,
    EvaluationConfig
)

# Find the best checkpoint
checkpoint_path = WORKING_PATH / 'checkpoints' / 'best_model.pt'
if not checkpoint_path.exists():
    # Find any .pt file
    pt_files = list((WORKING_PATH / 'checkpoints').glob('*.pt'))
    if pt_files:
        checkpoint_path = pt_files[0]
        print(f"Using checkpoint: {checkpoint_path}")
    else:
        raise FileNotFoundError("No checkpoint found!")
else:
    print(f"Using best checkpoint: {checkpoint_path}")

# Define evaluation configuration
eval_config = EvaluationConfig(
    checkpoint_path=str(checkpoint_path),
    model_path=str(RAW_DATA_PATH / 'layoutlm-base-uncased'),
    processed_data_path=str(WORKING_PATH / 'processed'),
    output_dir=str(WORKING_PATH / 'evaluation'),
    batch_size=32,  # Can use larger batch for inference
    run_fraud_detection=True
)

print("\nStarting evaluation...")
eval_summary = run_evaluation_pipeline(eval_config)

# Print results
ner_metrics = eval_summary.get('ner_metrics', {})
macro_f1 = ner_metrics.get('macro', {}).get('f1', 0.0)

print(f"\nEvaluation Results:")
print(f"Macro F1 Score: {macro_f1:.4f}")

# Per-entity scores
print("\nPer-Entity F1 Scores:")
for entity, metrics in ner_metrics.get('per_entity', {}).items():
    print(f"  {entity}: {metrics['f1']:.4f}")

# Fraud detection results
fraud_report = eval_summary.get('fraud_report', {})
if fraud_report.get('status') == 'completed':
    print(f"\nFraud Detection:")
    print(f"  Total processed: {fraud_report.get('total_processed', 0)}")
    print(f"  Duplicates found: {fraud_report.get('duplicates_found', 0)}")
    print(f"  Duplicate rate: {fraud_report.get('duplicate_rate', 0):.2%}")

## Cell 6: Save Output to /kaggle/working/

In [ ]:
import json
from datetime import datetime

# Create final summary report
final_report = {
    'timestamp': datetime.now().isoformat(),
    'kaggle_config': {
        'batch_size': 16,
        'num_epochs': 15,
        'device': str(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU',
        'gpu_count': torch.cuda.device_count()
    },
    'preprocessing': {
        'total_samples': preprocess_summary.get('total_samples', 0),
        'failed_samples': preprocess_summary.get('failed_samples_count', 0)
    },
    'training': {
        'status': training_summary.get('status'),
        'best_eval_loss': training_summary.get('best_eval_loss'),
        'best_checkpoint': training_summary.get('best_checkpoint'),
        'epochs_trained': training_summary.get('epochs_trained'),
        'training_time_seconds': training_summary.get('training_time_seconds')
    },
    'evaluation': {
        'macro_f1': ner_metrics.get('macro', {}).get('f1'),
        'per_entity_f1': {
            entity: metrics['f1']
            for entity, metrics in ner_metrics.get('per_entity', {}).items()
        },
        'duplicates_detected': fraud_report.get('duplicates_found', 0)
    },
    'output_paths': {
        'processed_data': str(WORKING_PATH / 'processed'),
        'checkpoints': str(WORKING_PATH / 'checkpoints'),
        'evaluation': str(WORKING_PATH / 'evaluation'),
        'best_model': str(checkpoint_path)
    }
}

# Save final report
report_path = WORKING_PATH / 'final_report.json'
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)
print(f"Final report saved to: {report_path}")

# Print summary of saved outputs
print("\n" + "="*60)
print("OUTPUT SUMMARY - All files saved to /kaggle/working/")
print("="*60)

for item in WORKING_PATH.rglob('*'):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        rel_path = item.relative_to(WORKING_PATH)
        print(f"  {rel_path} ({size_mb:.2f} MB)")

print("="*60)
print(f"\nBest model checkpoint: {checkpoint_path}")
print(f"Evaluation results: {WORKING_PATH / 'evaluation'}")
print(f"Final report: {report_path}")
print("\nAll outputs are preserved in /kaggle/working/ for download!")